In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_rows', None)

In [109]:
X_train = pd.read_csv('../../../data/processed/for_linear_model/FW,MF/predictors_train_filtered.csv')
X_test = pd.read_csv('../../../data/processed/for_linear_model/FW,MF/predictors_test_filtered.csv')
y_train = pd.read_csv('../../../data/processed/for_linear_model/FW,MF/target_train_filtered_logariphmed.csv')
y_test = pd.read_csv('../../../data/processed/for_linear_model/FW,MF/target_test_filtered_logariphmed.csv')

In [4]:
train_players = pd.read_csv('../../../data/processed/for_linear_model/FW,MF/train_players_names.csv')
test_players = pd.read_csv('../../../data/processed/for_linear_model/FW,MF/test_players_names.csv')

In [5]:
all_cols = X_train.columns
all_cols

Index(['Age', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', 'CrdY', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
       '2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'OG', 'Sh', 'SoT',
       'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1'],
      dtype='object')

<b>Регрессор</b>

In [97]:
LR_regressor = LinearRegression()
LR_regressor.fit(X_train, y_train)

LinearRegression()

<b>Предсказанные значения</b>

In [98]:
y_pred_log = LR_regressor.predict(X_test)
y_pred = np.exp(y_pred_log)
y_test_original = np.exp(y_test)

y_train_pred = np.exp(LR_regressor.predict(X_train))
y_train_original = np.exp(y_train)

<b>Метрики</b>

In [99]:
mae_test = mean_absolute_error(y_test_original, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2_test = r2_score(y_test_original, y_pred)
mape_test = mean_absolute_percentage_error(y_test_original, y_pred)

mae_train = mean_absolute_error(y_train_original, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train_original, y_train_pred))
r2_train = r2_score(y_train_original, y_train_pred)
mape_train = mean_absolute_percentage_error(y_train_original, y_train_pred)

metrics = pd.DataFrame({
    'MAE':[mae_test, mae_train],
    'RMSE':[rmse_test, rmse_train],
    'MAPE':[mape_test, mape_train],
    'R2':[r2_test, r2_train]
}, index = ['test', 'train'])

<b>Сравнение остатков</b>

In [100]:
y_pred = pd.Series(np.reshape(y_pred, len(y_pred)), name='Value_pred')
y_train_pred = pd.Series(np.reshape(y_train_pred, len(y_train_pred)), name='Value_pred')

comp_leftovers_test = pd.concat([test_players, y_pred, y_test_original], axis=1)
comp_leftovers_train = pd.concat([train_players, y_train_pred, y_train_original], axis=1)

<b>Коэффициенты</b>

In [101]:
coeffs = LR_regressor.coef_[0]
coeffs_df = pd.DataFrame({
    'coeff':coeffs.astype('float')
}, index=X_train.columns)
coeffs_df = coeffs_df.sort_values(by='coeff', key=abs, ascending=False)

<b>Все переменные</b>

In [10]:
coeffs_df

,coeff
Min,-1.459256
90s,1.043555
league_GB1,0.755900
G-PK.1,-0.611205
G+A-PK,0.535746
Starts,0.504469
Gls.1,0.480824
league_IT1,0.435443
G/Sh,-0.329266
Fls,-0.320049


In [28]:
metrics

,MAE,RMSE,MAPE,R2
test,4.382325,8.073346,0.833730,0.617946
train,5.075260,15.356006,0.672494,0.138782


In [29]:
comp_leftovers_test

,Player,Value_pred,Value
0,Antonio Nusa,8.143371,28.00
1,Luis Vazquez,2.025725,2.50
2,Tristan Degreef,0.919353,2.00
3,Maximilian Breunig,3.030932,1.50
4,Kike Barja,1.250319,1.50
5,Pontus Almqvist,3.380170,2.50
6,Jacob Ramsey,20.577263,32.00
7,Luciano Gondou,6.069575,12.00
8,Issiaka Kamate,0.765947,0.75
9,Lucas Beltran,5.981920,18.00


In [30]:
comp_leftovers_train

,Player,Value_pred,Value
0,Vincenzo Grifo,20.035632,6.000
1,Rifat Zhemaletdinov,1.259427,2.500
2,Roger Fernandes,4.364004,22.000
3,Gyrano Kerk,2.358704,3.000
4,Musa Al-Taamari,5.404008,6.000
5,Emilio Kehrer,1.062651,0.500
6,Albert Gudmundsson,4.620049,18.000
7,Moses Simon,8.303744,12.000
8,Michal Skoras,0.758983,2.500
9,Sirlord Conteh,1.599808,1.500


<b>Модель 1.<br>
Переменные: 'G+A', 'Ast', 'SoT', 'G+A-PK', '90s', 'Ast.1', 'Fld', 'Crs', 'SoT/90', 'Off', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1'</b>

In [31]:
X_train = X_train[['G+A', 'Ast', 'SoT', 'G+A-PK', '90s', 'Ast.1', 'Fld', 'Crs', 'SoT/90', 'Off', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'Ast', 'SoT', 'G+A-PK', '90s', 'Ast.1', 'Fld', 'Crs', 'SoT/90', 'Off', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [37]:
coeffs_df

,coeff
league_GB1,0.735027
SoT,0.439562
league_IT1,0.373715
league_L1,0.276272
league_FR1,0.273967
league_ES1,0.228564
G+A-PK,0.204463
league_NL1,-0.181836
Ast,0.171821
Fld,0.109514


In [38]:
metrics

,MAE,RMSE,MAPE,R2
test,5.010587,10.142198,0.790099,0.397050
train,5.396066,13.178744,0.824118,0.365686


In [39]:
comp_leftovers_test

,Player,Value_pred,Value
0,Antonio Nusa,3.712396,28.00
1,Luis Vazquez,1.880706,2.50
2,Tristan Degreef,0.604785,2.00
3,Maximilian Breunig,1.893953,1.50
4,Kike Barja,1.423746,1.50
5,Pontus Almqvist,3.632810,2.50
6,Jacob Ramsey,12.489325,32.00
7,Luciano Gondou,4.012628,12.00
8,Issiaka Kamate,0.527373,0.75
9,Lucas Beltran,7.422511,18.00


In [40]:
comp_leftovers_train

,Player,Value_pred,Value
0,Vincenzo Grifo,28.678508,6.000
1,Rifat Zhemaletdinov,1.532433,2.500
2,Roger Fernandes,3.698332,22.000
3,Gyrano Kerk,3.592852,3.000
4,Musa Al-Taamari,4.440971,6.000
5,Emilio Kehrer,0.865675,0.500
6,Albert Gudmundsson,5.931311,18.000
7,Moses Simon,14.716896,12.000
8,Michal Skoras,0.810240,2.500
9,Sirlord Conteh,2.217511,1.500


<b>Модель 2.<br>Переменные: 'G+A', 'Ast', 'SoT', 'G+A-PK', '90s', 'Ast.1', 'Fld', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1'</b>

In [41]:
X_train = X_train[['G+A', 'Ast', 'SoT', 'G+A-PK', '90s', 'Ast.1', 'Fld', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'Ast', 'SoT', 'G+A-PK', '90s', 'Ast.1', 'Fld', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [47]:
coeffs_df

,coeff
league_GB1,0.741435
SoT,0.427203
league_IT1,0.381107
league_FR1,0.277803
league_L1,0.276529
league_ES1,0.226258
G+A-PK,0.204598
league_NL1,-0.173222
Ast,0.160384
G+A,0.112886


In [48]:
metrics

,MAE,RMSE,MAPE,R2
test,4.904375,9.684149,0.786559,0.450282
train,5.342185,13.165542,0.824999,0.366956


In [49]:
comp_leftovers_test

,Player,Value_pred,Value
0,Antonio Nusa,3.747776,28.00
1,Luis Vazquez,1.910895,2.50
2,Tristan Degreef,0.587454,2.00
3,Maximilian Breunig,1.852731,1.50
4,Kike Barja,1.413495,1.50
5,Pontus Almqvist,3.509293,2.50
6,Jacob Ramsey,12.134180,32.00
7,Luciano Gondou,3.993957,12.00
8,Issiaka Kamate,0.523586,0.75
9,Lucas Beltran,6.970824,18.00


In [52]:
comp_leftovers_train

,Player,Value_pred,Value
0,Vincenzo Grifo,27.510818,6.000
1,Rifat Zhemaletdinov,1.489575,2.500
2,Roger Fernandes,3.704442,22.000
3,Gyrano Kerk,3.994239,3.000
4,Musa Al-Taamari,4.374957,6.000
5,Emilio Kehrer,0.849651,0.500
6,Albert Gudmundsson,5.832401,18.000
7,Moses Simon,16.211570,12.000
8,Michal Skoras,0.812585,2.500
9,Sirlord Conteh,2.199063,1.500


<b>Модель 3.<br>Переменные: 'Gls', 'Ast', 'SoT', 'G+A-PK', '90s', 'Fld', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1'</b>

In [78]:
X_train = X_train[['Gls', 'Ast', 'SoT', 'G+A-PK', '90s', 'Fld', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Gls', 'Ast', 'SoT', 'G+A-PK', '90s', 'Fld', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [73]:
coeffs_df

,coeff
league_GB1,0.759883
league_IT1,0.425914
SoT,0.398776
Age,-0.308773
league_L1,0.302773
league_ES1,0.294168
league_FR1,0.274001
Ast,0.213958
league_NL1,-0.170033
G+A-PK,0.143286


In [82]:
metrics

,MAE,RMSE,MAPE,R2
test,4.931770,10.741656,0.804759,0.323668
train,5.066766,12.299104,0.741155,0.447537


In [63]:
comp_leftovers_test

,Player,Value_pred,Value
0,Antonio Nusa,6.280616,28.00
1,Luis Vazquez,1.938061,2.50
2,Tristan Degreef,0.829723,2.00
3,Maximilian Breunig,2.096281,1.50
4,Kike Barja,1.362094,1.50
5,Pontus Almqvist,3.651289,2.50
6,Jacob Ramsey,13.810790,32.00
7,Luciano Gondou,4.442946,12.00
8,Issiaka Kamate,0.807218,0.75
9,Lucas Beltran,8.347471,18.00


In [64]:
comp_leftovers_train

,Player,Value_pred,Value
0,Vincenzo Grifo,18.894152,6.000
1,Rifat Zhemaletdinov,1.241305,2.500
2,Roger Fernandes,5.605908,22.000
3,Gyrano Kerk,2.939458,3.000
4,Musa Al-Taamari,3.297011,6.000
5,Emilio Kehrer,0.985541,0.500
6,Albert Gudmundsson,5.311651,18.000
7,Moses Simon,11.354530,12.000
8,Michal Skoras,0.775369,2.500
9,Sirlord Conteh,1.753462,1.500


<b>Модель 4.<br>
Переменные: 'Gls', 'Ast', 'G+A-PK', '90s', 'Fld', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1'</b>

In [96]:
X_train = X_train[['Gls', 'Ast', 'G+A-PK', '90s', 'Fld', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Gls', 'Ast', 'G+A-PK', '90s', 'Fld', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [103]:
coeffs_df

,coeff
league_GB1,0.712847
Gls,0.367860
league_IT1,0.361012
Age,-0.315112
league_L1,0.260582
90s,0.240181
league_ES1,0.239881
league_NL1,-0.239678
Ast,0.229201
league_FR1,0.219051


In [104]:
metrics

,MAE,RMSE,MAPE,R2
test,4.438201,8.014056,0.817920,0.623537
train,5.165309,12.884933,0.763453,0.393654


In [105]:
comp_leftovers_test

,Player,Value_pred,Value
0,Antonio Nusa,7.780520,28.00
1,Luis Vazquez,1.920413,2.50
2,Tristan Degreef,1.039289,2.00
3,Maximilian Breunig,2.366052,1.50
4,Kike Barja,1.402858,1.50
5,Pontus Almqvist,3.111401,2.50
6,Jacob Ramsey,17.643419,32.00
7,Luciano Gondou,4.775285,12.00
8,Issiaka Kamate,0.866643,0.75
9,Lucas Beltran,11.763245,18.00


In [107]:
comp_leftovers_train

,Player,Value_pred,Value
0,Vincenzo Grifo,15.918552,6.000
1,Rifat Zhemaletdinov,1.329983,2.500
2,Roger Fernandes,3.528582,22.000
3,Gyrano Kerk,3.011677,3.000
4,Musa Al-Taamari,3.077539,6.000
5,Emilio Kehrer,0.854593,0.500
6,Albert Gudmundsson,5.280265,18.000
7,Moses Simon,16.245926,12.000
8,Michal Skoras,0.784917,2.500
9,Sirlord Conteh,1.862312,1.500


In [ ]:
base_cols = ['Gls', 'Ast', 'G+A-PK', '90s', 'Fld', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']

In [110]:
cols_to_add = [col for col in all_cols if not(col in base_cols)]
new_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_add:
    X_train_new = X_train[base_cols + [col]]
    X_test_new = X_test[base_cols + [col]]
    LR_regressor.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new<mae_test or mape_new<mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=new_cols.columns, index=[col])
        new_cols = pd.concat([new_cols, row])
new_cols

,mae,mape,r2
base,4.438201,0.81792,0.623537


In [111]:
cols_to_delete = [col for col in base_cols if not(col.startswith('league'))]
del_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_delete:
    X_train_new = X_train[[c for c in base_cols if c != col]]
    X_test_new = X_test[[c for c in base_cols if c != col]]
    LR_regressor.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new < mae_test or mape_new < mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=del_cols.columns, index=[col])
        del_cols = pd.concat([del_cols, row])
del_cols

,mae,mape,r2
base,4.438201,0.81792,0.623537
